# High-Dimensional Geometry and the Concentration of Distances — companion notebook

This notebook is the **narrative** half of the high-dimensional-geometry Python pillar. The
reusable, tested implementation lives next door in [`high_dimensional_geometry.py`](high_dimensional_geometry.py);
here we import it and walk the topic section by section, so every claim the page makes renders as
executed output.

**One source of truth.** `high_dimensional_geometry.py` owns the numbers — its `assert`-based harness
encodes the thin shell, near-orthogonality, distance concentration, volume concentration, and the
intrinsic-dimension resolution — and its `grid_table()` is mirrored *to the decimal* by
`ConcentrationLaboratory.tsx`. This notebook never redefines that math; it calls into it.

Run from the repo root:

```
uv run --with numpy --with scipy --with jupyter jupyter execute notebooks/high-dimensional-geometry/01_high_dimensional_geometry.ipynb
```

or run the reference implementation directly, which executes the same checks:

```
uv run --with numpy --with scipy python notebooks/high-dimensional-geometry/high_dimensional_geometry.py
```

In [ ]:
import sys
import pathlib

# Make high_dimensional_geometry.py importable whether the kernel starts in the repo
# root or in this notebook's own directory.
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "high-dimensional-geometry", pathlib.Path("notebooks/high-dimensional-geometry")):
    if (_cand / "high_dimensional_geometry.py").exists():
        sys.path.insert(0, str(_cand))
        break

from high_dimensional_geometry import (
    GRID,
    sample_gaussian, sample_sphere, sample_cube,
    norm_concentration_stats, inner_product_stats,
    relative_contrast, squared_distance_relative_variance,
    ball_volume, shell_fraction,
    structured_data, iid_data, twonn_intrinsic_dim,
    grid_table, finance_demo,
    test_chi_square_moments, test_norm_concentration, test_near_orthogonality,
    test_distance_concentration, test_relative_variance_vanishes,
    test_volume_concentration, test_intrinsic_dimension,
)

## 1. The thin shell: the norm of a random vector concentrates

For $x \sim \mathcal N(0, I_d)$ the squared norm $\lVert x\rVert^2 = \sum_i x_i^2$ is a sum of $d$
independent $\chi^2_1$ terms, so $\mathbb E\lVert x\rVert^2 = d$ and $\operatorname{Var}\lVert x\rVert^2 = 2d$.
The relative spread of $\lVert x\rVert/\sqrt d$ therefore shrinks like $1/\sqrt d$: almost all the mass
of a high-dimensional Gaussian sits on a thin shell of radius $\sqrt d$, nowhere near the origin where
its density is largest.

In [ ]:
print(f"{'d':>6}{'mean ||x||^2/d':>18}{'var ||x||^2/d':>16}{'2/d':>10}{'shell std':>12}")
for d in (1, 10, 100, 1000):
    s = norm_concentration_stats(d, n=8000, seed=2)
    print(f"{d:>6}{s['mean_sqnorm_over_d']:>18.4f}{s['var_sqnorm_over_d']:>16.5f}{2/d:>10.5f}{s['shell_std']:>12.4f}")
print()
test_chi_square_moments()
test_norm_concentration()

## 2. Near-orthogonality of random vectors

Two independent unit vectors $u, v$ have $\mathbb E\langle u, v\rangle = 0$ and
$\operatorname{Var}\langle u, v\rangle = 1/d$. As $d$ grows their inner product concentrates at $0$ —
the angle between independent random directions approaches $90^\circ$. There is room in high
dimensions for exponentially many nearly orthogonal directions.

In [ ]:
print(f"{'d':>6}{'mean <u,v>':>14}{'var <u,v>':>14}{'1/d':>10}{'mean |<u,v>|':>16}")
for d in (2, 10, 100, 1000):
    s = inner_product_stats(d, n=8000, seed=0)
    print(f"{d:>6}{s['mean']:>14.4f}{s['var']:>14.5f}{1/d:>10.5f}{s['mean_abs']:>16.4f}")
print()
test_near_orthogonality()

## 3. The concentration of distances (the curse)

For data with i.i.d. coordinates the relative variance of squared distance,
$\operatorname{Var}(D^2)/\mathbb E[D^2]^2$, vanishes — exactly $2/d$ for Gaussian data — so every
pairwise distance concentrates at a common value. The **relative contrast**
$(D_{\max}-D_{\min})/D_{\min}$ of a query's distances then collapses toward $0$ (Beyer et al. 1999):
the nearest and farthest neighbors become indistinguishable, and exact nearest-neighbor search loses
meaning.

In [ ]:
print(f"{'d':>6}{'rel. contrast':>16}{'Var(D^2)/E[D^2]^2':>20}{'2/d':>10}")
for d in (2, 10, 100, 1000):
    c = relative_contrast(d, seed=10)
    rv = squared_distance_relative_variance(d, seed=0)
    print(f"{d:>6}{c:>16.4f}{rv:>20.5f}{2/d:>10.5f}")
print()
test_distance_concentration()
test_relative_variance_vanishes()

## 4. Volume in high dimensions

The volume of the unit ball, $V_d = \pi^{d/2}/\Gamma(d/2+1)$, peaks at $d=5$ and then collapses to
$0$. What volume remains flees to the surface: the fraction of the ball within $\varepsilon$ of the
boundary is $1-(1-\varepsilon)^d \to 1$. A high-dimensional orange is almost all peel — and, by the
same concentration, almost all of a sphere's area lies near any equator.

In [ ]:
print(f"{'d':>6}{'ball volume':>16}{'mass in outer 10%':>20}")
for d in (2, 5, 10, 20, 100, 1000):
    print(f"{d:>6}{ball_volume(d):>16.6g}{shell_fraction(d, 0.10):>20.4f}")
print()
test_volume_concentration()

## 5. Why retrieval still works: intrinsic dimension

The curse assumes data that fill $\mathbb R^d$. Real embeddings do not — they lie near a
low-dimensional manifold, and contrast is governed by the *intrinsic* dimension $k$, not the ambient
$d$. The TwoNN estimator (Facco et al. 2017) recovers $k$ from each point's two nearest-neighbor
distances. Structured data with $k=10$ embedded in $\mathbb R^{200}$ reads TwoNN $\approx 10$, and
i.i.d. data in $\mathbb R^{10}$ reads $\approx 10$ too — the estimator sees the dimension the data
actually occupies.

In [ ]:
k_struct = twonn_intrinsic_dim(structured_data(1000, 200, k=10, noise=0.01, seed=0))
k_amb = twonn_intrinsic_dim(iid_data(1000, 10, seed=1))
print(f"TwoNN, structured k=10 in R^200 : {k_struct:.2f}")
print(f"TwoNN, i.i.d. in R^10           : {k_amb:.2f}")
print()
test_intrinsic_dimension()

## 6. The grid the viz mirrors

`grid_table()` is the single source of the numbers `ConcentrationLaboratory.tsx` displays. Each row
is one dimension on the slider; the columns are the relative contrast, the inner-product variance
against $1/d$, the shell width $\operatorname{std}(\lVert x\rVert/\sqrt d)$, and the outer-shell mass.

In [ ]:
print(f"{'d':>6}{'contrast':>12}{'Var<u,v>':>12}{'1/d':>10}{'shell_std':>12}{'shell10%':>11}")
for r in grid_table():
    print(f"{int(r['d']):>6}{r['contrast']:>12.4f}{r['ip_var']:>12.5f}{r['ip_var_theory']:>10.5f}{r['shell_std']:>12.4f}{r['shell_frac_10pct']:>11.4f}")

## 7. Finance case study: is cosine meaningful at 1536 dimensions?

Production financial-document retrieval embeds 10-K passages and earnings-call chunks into
$\mathbb R^{1536}$. Given §§1–3, is cosine similarity meaningful there? Yes — because
financial-document embeddings have *low intrinsic dimension*. A synthetic stand-in (a
low-rank-plus-noise set, not a trained encoder) keeps its contrast at $d=1536$ where i.i.d. data
destroys it, and TwoNN recovers the small intrinsic dimension of the structured set.

In [ ]:
finance_demo()

---

Every numerical claim above is also an `assert` in
[`high_dimensional_geometry.py`](high_dimensional_geometry.py), so running that file as a script is a
regression test for the whole topic:

```
uv run --with numpy --with scipy python notebooks/high-dimensional-geometry/high_dimensional_geometry.py
```

and `ConcentrationLaboratory.tsx` reproduces the `grid_table()` numbers to the decimal. The three
pillars — math, viz, and code — agree by construction.